# SkinScope AI — Real Clinical Dataset & Production ML Pipeline (Google Colab)

This notebook demonstrates both components of SkinScope AI's ML architecture:
1. **Part 1 — Real Public Clinical Dataset (UCI Dermatology Dataset):** 366 real clinical patient records, 34 clinical & histopathological features (including `erythema`, `scaling`, `itching`, `melanin_incontinence`, `age`), 6 dermatological classes (`Psoriasis`, `Seboreic Dermatitis`, `Lichen Planus`, `Pityriasis Rosea`, `Chronic Dermatitis`, `Pityriasis Rubra Pilaris`) from the **UCI Machine Learning Repository**.
2. **Part 2 — SkinScope AI Production Chromatic Vision ML Engine:** 1:1 Python implementation of `src/lib/color-analysis.ts` (`RGB -> HSV + YCbCr` biological tissue gatekeeper, bilirubin/jaundice & erythema/burn indices, and region-calibrated classifier).

## Part 1: Train & Evaluate ML Models on the Real UCI Dermatology Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

CLASS_NAMES = {
    1: 'Psoriasis',
    2: 'Seboreic Dermatitis',
    3: 'Lichen Planus',
    4: 'Pityriasis Rosea',
    5: 'Chronic Dermatitis',
    6: 'Pityriasis Rubra Pilaris',
}

URL = 'https://raw.githubusercontent.com/STEVEN-197/Skin-Scope-Ai/main/ml-colab/uci_dermatology_dataset.csv'
df = pd.read_csv(URL, na_values=['?'])
df['age'] = df['age'].fillna(df['age'].median())

print(f'Loaded Real UCI Dermatology Dataset: {df.shape[0]} real patient records, {df.shape[1] - 1} clinical features')
print('\nClass distribution:')
print(df['class'].map(CLASS_NAMES).value_counts())
df.head()

In [ ]:
X = df.drop(columns=['class'])
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Random Forest (150 trees)': RandomForestClassifier(n_estimators=150, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
}

best_name, best_model, best_acc = None, None, -1.0
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    cv = cross_val_score(model, X, y, cv=5).mean()
    print(f'{name:28s} | Test Accuracy: {acc*100:6.2f}% | 5-Fold CV: {cv*100:6.2f}%')
    if acc > best_acc:
        best_name, best_model, best_acc = name, model, acc

print(f'\nBest Model: {best_name} ({best_acc*100:.2f}% test accuracy)\n')
y_pred = best_model.predict(X_test)
target_names = [CLASS_NAMES[c] for c in sorted(y.unique())]
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=target_names, xticks_rotation=35, ax=axes[0], cmap='Blues'
)
axes[0].set_title(f'Confusion Matrix ({best_name})')

rf = models['Random Forest (150 trees)']
top10 = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10).sort_values()
top10.plot(kind='barh', ax=axes[1], color='#3b82f6')
axes[1].set_title('Top 10 Clinical Features (Random Forest)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## Part 2: SkinScope AI Live Chromatic Vision ML Engine (`src/lib/color-analysis.ts`)
Exact Python port of the production image feature extractor, biological tissue gatekeeper (`HSV + YCbCr`), and region-calibrated classifier (`eye`, `palm`, `skin`).

In [ ]:
from PIL import Image
import math

def rgb_to_hsv(r, g, b):
    rf, gf, bf = r / 255.0, g / 255.0, b / 255.0
    mx, mn = max(rf, gf, bf), min(rf, gf, bf)
    d = mx - mn
    h = 0.0
    s = 0.0 if mx == 0 else d / mx
    v = mx
    if d != 0:
        if mx == rf: h = ((gf - bf) / d) % 6
        elif mx == gf: h = (bf - rf) / d + 2
        else: h = (rf - gf) / d + 4
        h *= 60.0
        if h < 0: h += 360.0
    return h, s * 100.0, v * 100.0

def rgb_to_ycbcr(r, g, b):
    y = 0.299 * r + 0.587 * g + 0.114 * b
    cb = 128 - 0.168736 * r - 0.331264 * g + 0.5 * b
    cr = 128 + 0.5 * r - 0.418688 * g - 0.081312 * b
    return y, cb, cr

def skinscope_analyze_image(img: Image.Image, region: str = 'skin'):
    img = img.convert('RGB')
    img.thumbnail((256, 256))
    pixels = list(img.getdata())
    n = max(1, len(pixels))
    skin = synth = sclera = yellow = red = dark = 0
    sum_r = sum_g = sum_b = sum_v = 0.0
    for r, g, b in pixels:
        sum_r += r; sum_g += g; sum_b += b
        hh, ss, vv = rgb_to_hsv(r, g, b)
        _, cb, cr = rgb_to_ycbcr(r, g, b)
        sum_v += vv
        gr = g / r if r > 0 else 1.0
        is_synth = (r > 120 and ((gr < 0.28 and ss > 65) or (ss > 82 and gr < 0.35) or cr > 202)) or (85 <= hh <= 315 and ss > 28 and vv > 20)
        if is_synth: synth += 1
        is_skin = (not is_synth) and r >= 35 and g >= 20 and b >= 14 and r >= g*0.90 and r >= b*0.88 and 0.28 <= gr <= 0.98 and (hh <= 72 or hh >= 315) and 3 <= ss <= 80 and 65 <= cb <= 142 and 122 <= cr <= 198
        if is_skin: skin += 1
        is_sclera = (not is_synth) and vv >= 48 and ss <= 42 and r >= 115 and g >= 110 and b >= 75
        if is_sclera: sclera += 1
        if is_skin or is_sclera:
            if 38 <= hh <= 72 and gr >= 0.80 and ss >= 18 and vv >= 42 and ((r + g) * 0.5 - b) >= 25:
                yellow += 1
            if (not is_synth) and (hh <= 14 or hh >= 340) and ss >= 30 and vv >= 25 and 0.28 <= gr <= 0.64 and (r - g) >= 42 and r >= 130:
                red += 1
        if vv < 18: dark += 1

    f = {
        'region': region,
        'skinTissueRatio': round(skin / n * 100, 2),
        'syntheticRatio': round(synth / n * 100, 2),
        'scleraRatio': round(sclera / n * 100, 2),
        'yellowRatio': round(yellow / n * 100, 2),
        'redRatio': round(red / n * 100, 2),
        'darkRatio': round(dark / n * 100, 2),
        'brightness': round(sum_v / n, 2),
    }
    # Tissue Gatekeeper
    if f['syntheticRatio'] > 35 and f['syntheticRatio'] > f['skinTissueRatio']:
        return {**f, 'valid': False, 'condition': 'Rejected: synthetic/non-biological object', 'severity': 'invalid'}
    if region != 'eye' and f['skinTissueRatio'] < 10:
        return {**f, 'valid': False, 'condition': 'Rejected: insufficient human tissue', 'severity': 'invalid'}

    y_th = {'mild': 1.8, 'moderate': 4.0, 'severe': 6.5} if region == 'eye' else ({'mild': 4.0, 'moderate': 9.0, 'severe': 16.0} if region == 'palm' else {'mild': 5.0, 'moderate': 11.0, 'severe': 20.0})
    r_th = {'mild': 2.0, 'moderate': 5.0, 'severe': 10.0} if region == 'eye' else {'mild': 2.5, 'moderate': 7.5, 'severe': 16.0}
    y, r = f['yellowRatio'], f['redRatio']
    condition, severity, confidence = 'Healthy appearance', 'none', 88
    if y >= y_th['severe']: condition, severity, confidence = 'Possible jaundice (severe)', 'severe', 91
    elif y >= y_th['moderate']: condition, severity, confidence = 'Possible jaundice (moderate)', 'moderate', 84
    elif y >= y_th['mild']: condition, severity, confidence = 'Possible jaundice (mild)', 'mild', 76
    elif r >= r_th['severe'] and f['darkRatio'] < 25: condition, severity, confidence = 'Possible burn or severe inflammation', 'severe', 88
    elif r >= r_th['moderate']: condition, severity, confidence = 'Possible burn, rash, or inflammation', 'moderate', 80
    elif r >= r_th['mild']: condition, severity, confidence = 'Mild redness or skin irritation', 'mild', 74
    return {**f, 'valid': True, 'condition': condition, 'severity': severity, 'confidence': confidence}

# Quick live test on 3 synthetic test patches (healthy skin, jaundiced eye, red plastic bucket)
tests = [
    ('Healthy Skin Patch', Image.new('RGB', (120, 120), (210, 160, 130)), 'skin'),
    ('Jaundiced Sclera Patch', Image.new('RGB', (120, 120), (225, 210, 95)), 'eye'),
    ('Synthetic Blue Bucket', Image.new('RGB', (120, 120), (20, 90, 220)), 'skin'),
]
for label, im, reg in tests:
    res = skinscope_analyze_image(im, reg)
    print(f"{label:24s} -> valid={res['valid']} | condition={res['condition']} | severity={res['severity']} | yellow={res['yellowRatio']}% | skin={res['skinTissueRatio']}%")